In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


In [2]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 10

## load and prepare data

In [3]:
df = pd.read_csv('cleaned_marketing_data.csv')

In [4]:
df.shape

(6240, 18)

In [5]:
f"{df.time.min()} to {df.time.max()}"

'2021-01-25 to 2024-01-15'

In [6]:
df.geo.nunique()

40

In [7]:
f"Total revenue: ${df.total_revenue.sum():,.2f}"

'Total revenue: $1,318,699,437.74'

In [8]:
df.dtypes

geo                             object
time                            object
Channel0_impression              int64
Channel1_impression              int64
Channel2_impression              int64
Channel3_impression              int64
competitor_sales_control       float64
sentiment_score_control        float64
Channel0_spend                 float64
Channel1_spend                 float64
Channel2_spend                 float64
Channel3_spend                 float64
Organic_channel0_impression      int64
Promo                          float64
conversions                    float64
revenue_per_conversion         float64
population                     float64
total_revenue                  float64
dtype: object

In [9]:
df['time'] = pd.to_datetime(df.time)

In [10]:
df = df.sort_values(['geo', 'time']).reset_index(drop=True)

In [11]:
df['month'] = df.time.dt.month
df['quarter'] = df.time.dt.quarter
df['year'] = df.time.dt.year
df['week_of_year'] = df.time.dt.isocalendar().week

In [12]:
df.head()

,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,...,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population,total_revenue,month,quarter,year,week_of_year
0,Geo0,2021-01-25,280668,0,0,470611,-1.338765,0.115581,2058.0608,0.00000,...,97320,0.000000,1954576.8,0.020055,136670.94,39198.556898,1,1,2021,4
1,Geo0,2021-02-01,366206,182108,19825,527702,0.893645,0.944224,2685.2874,1755.74540,...,201441,0.000000,2064249.6,0.020103,136670.94,41497.960631,2,1,2021,5
2,Geo0,2021-02-08,197565,230170,0,393618,-0.284549,-1.290579,1448.6895,2219.12230,...,0,0.683819,2086382.8,0.019929,136670.94,41579.088854,2,1,2021,6
3,Geo0,2021-02-15,140990,66643,0,326034,-1.034740,-1.084514,1033.8406,642.52057,...,0,1.289055,2826431.5,0.019987,136670.94,56492.861509,2,1,2021,7
4,Geo0,2021-02-22,399116,164991,0,381982,-0.319276,-0.017503,2926.6072,1590.71640,...,0,0.227739,3551929.2,0.020000,136670.94,71039.827175,2,1,2021,8


In [13]:
month_dummies = pd.get_dummies(df.month, prefix='month', drop_first=True)
df = pd.concat([df, month_dummies], axis=1)

In [14]:
month_dummies

,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,False,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
6235,False,False,False,False,False,False,False,False,False,False,True
6236,False,False,False,False,False,False,False,False,False,False,True
6237,False,False,False,False,False,False,False,False,False,False,False
6238,False,False,False,False,False,False,False,False,False,False,False


In [15]:
df.head()

,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,Geo0,2021-01-25,280668,0,0,470611,-1.338765,0.115581,2058.0608,0.00000,...,False,False,False,False,False,False,False,False,False,False
1,Geo0,2021-02-01,366206,182108,19825,527702,0.893645,0.944224,2685.2874,1755.74540,...,False,False,False,False,False,False,False,False,False,False
2,Geo0,2021-02-08,197565,230170,0,393618,-0.284549,-1.290579,1448.6895,2219.12230,...,False,False,False,False,False,False,False,False,False,False
3,Geo0,2021-02-15,140990,66643,0,326034,-1.034740,-1.084514,1033.8406,642.52057,...,False,False,False,False,False,False,False,False,False,False
4,Geo0,2021-02-22,399116,164991,0,381982,-0.319276,-0.017503,2926.6072,1590.71640,...,False,False,False,False,False,False,False,False,False,False


In [16]:
# create temporal time split

# unique time period
unique_times = df.time.unique()
n_times = len(unique_times)

# calculate time split point
test_size = 0.2
split_index = int(n_times * (1 - test_size))
split_date = unique_times[split_index]
split_date

Timestamp('2023-06-12 00:00:00')

In [17]:
train_df = df[df.time < split_date].copy()
test_df = df[df.time >= split_date].copy()

In [18]:
print(f"{train_df.time.min()} to {train_df.time.max()}")
print(f"{test_df.time.min()} to {test_df.time.max()}")
print(f"Train revenue: ${train_df.total_revenue.sum():,.2f}")
print(f"Test revenue: ${test_df.total_revenue.sum():,.2f}")

2021-01-25 00:00:00 to 2023-06-05 00:00:00
2023-06-12 00:00:00 to 2024-01-15 00:00:00
Train revenue: $1,044,275,572.82
Test revenue: $274,423,864.92


In [19]:
df_engineered = df.copy()
media_channels = ['Channel0', 'Channel1', 'Channel2', 'Channel3']

In [25]:
spend_cols = [f'{ch}_spend' for ch in media_channels]
spend_cols = df_engineered[spend_cols]

In [27]:
spend_cols.describe()

,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend
count,6240.000000,6240.000000,6240.000000,6240.000000
mean,6490.659147,5019.278558,1935.793164,14080.185617
std,6008.031135,5983.571424,4125.531614,10168.239254
min,0.000000,0.000000,0.000000,0.000000
25%,1859.914025,0.000000,0.000000,6299.393250
50%,4983.835500,2995.072650,0.000000,11731.178500
75%,9709.803000,7785.800675,1892.909150,19997.351500
max,38071.730000,42397.700000,41688.645000,59499.484000
